# Notebook 14 – Scikit-learn Preprocessing Pipeline



In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.DataFrame({
    'Age': [24, 31, np.nan, 45, 29, 38],
    'Salary': [25000, np.nan, 52000, 75000, 40000, 62000],
    'City': ['Hassan', 'Bengaluru', 'Hassan', None, 'Mysuru', 'Bengaluru'],
    'Department': ['IT', 'Sales', None, 'IT', 'HR', 'Sales']
})
print(df)

    Age   Salary       City Department
0  24.0  25000.0     Hassan         IT
1  31.0      NaN  Bengaluru      Sales
2   NaN  52000.0     Hassan       None
3  45.0  75000.0       None         IT
4  29.0  40000.0     Mysuru         HR
5  38.0  62000.0  Bengaluru      Sales


## 1. Why use Pipelines?

### Definition
A pipeline organizes preprocessing into repeatable steps. It reduces manual work and inconsistent transformations. It applies the same sequence to future records. It also helps prevent train-test contamination.

### Business Example
A company cleans employee salary and city data every month. Repeating manual operations can introduce mistakes. A pipeline applies the same rules each time. New employees receive consistent processing.

### AI/ML Use
A model needs the same transformations during training and prediction. Pipelines store fitted parameters such as medians and scaling values. They work well with cross-validation. They make deployment easier.

### Python Example

In [3]:
print("Missing values before:", df.isna().sum().to_dict())

Missing values before: {'Age': 1, 'Salary': 1, 'City': 1, 'Department': 1}


### Explanation
The counts show which columns need cleaning. A pipeline can perform these operations consistently. We do not have to remember each manual step. It is especially useful when new data arrives.

## 2. Pipeline

### Definition
`Pipeline` connects ordered transformations. Each intermediate step must support `fit` and `transform`. The last step can be a transformer or estimator. The pipeline executes steps in the given order.

### Business Example
An HR analyst fills missing salaries and then scales them. Those steps must happen in order. A pipeline makes that order explicit. It can be reused for other employee records.

### AI/ML Use
Training folds should learn imputation and scaling only from their training rows. A pipeline supports that pattern. It prevents repeated code. It also simplifies model tuning.

### Python Example

In [5]:
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
print(num_pipe)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])


### Explanation
The imputer replaces missing numbers with a median. The scaler then standardizes the completed numbers. The order matters because scaling needs numeric values. Both steps are packaged in one object.

## 3. ColumnTransformer

### Definition
`ColumnTransformer` applies different transformations to different columns. Numerical and categorical fields need different processing. It combines the outputs into one feature matrix. Unselected columns can be dropped or passed through.

### Business Example
Salary and age are numeric, while city is categorical. A business cannot scale city names directly. Different rules are required for each group. ColumnTransformer handles both together.

### AI/ML Use
Most ML datasets mix numerical and categorical features. ColumnTransformer processes each group safely. It works inside a Pipeline. The output can be passed to a classifier or regressor.

### Python Example

In [7]:
print("Numeric:", df.select_dtypes(include="number").columns.tolist())
print("Categorical:", df.select_dtypes(exclude="number").columns.tolist())

Numeric: ['Age', 'Salary']
Categorical: ['City', 'Department']


### Explanation
The code separates columns by data type. This tells us where to apply numeric and categorical transformations. We will combine them in the complete example. The selected columns should reflect actual feature meanings.

## 4. Numerical Pipeline

### Definition
A numerical pipeline prepares continuous or count features. Missing values must be handled before most scalers. Scaling changes numerical magnitude but not the column meaning. Different scalers suit different distributions.

### Business Example
An employee dataset includes age and salary. Missing salaries can be filled with the median. Values can then be standardized. The result is suitable for distance-sensitive models.

### AI/ML Use
Algorithms such as KNN and logistic regression often benefit from scaling. Imputation ensures there are no NaN values. The pipeline learns parameters from training data. The same parameters are reused for testing.

### Python Example

In [9]:
numeric_cols = ["Age", "Salary"]
print(num_pipe.fit_transform(df[numeric_cols]).round(2))

[[-1.33 -1.65]
 [-0.3   0.06]
 [-0.3   0.06]
 [ 1.78  1.52]
 [-0.59 -0.7 ]
 [ 0.74  0.7 ]]


### Explanation
The first step fills missing values with column medians. The second step gives each column approximately zero mean and unit variance. These results are numeric arrays. Fit only on training data in a real ML project.

## 5. Categorical Pipeline

### Definition
A categorical pipeline prepares text labels for ML. Missing categories can be filled with a common value. Encoding converts labels into numeric columns. Unknown categories require a safe policy.

### Business Example
City and department are labels, not quantities. Missing labels can be marked Unknown. One-hot encoding creates indicator columns. A model can then use them without assuming an order.

### AI/ML Use
One-hot encoding is useful for nominal features. The imputer must run before the encoder. `handle_unknown="ignore"` avoids errors for unseen labels. High-cardinality features may require other encoders.

### Python Example

In [11]:
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")), ("encoder", OneHotEncoder(handle_unknown="ignore"))])
print(cat_pipe.fit_transform(df[["City", "Department"]]).toarray())

[[0. 1. 0. 0. 0. 1. 0. 0.]
 [1. 0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0. 1. 0. 0.]
 [0. 0. 1. 0. 1. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 1. 0.]]


### Explanation
Missing categories become Unknown. The encoder creates separate indicator columns. Each row is represented numerically. Unseen categories will not cause an exception at prediction time.

## 6. Imputation Pipeline

### Definition
Imputation replaces missing feature values using a chosen rule. Median works for many skewed numeric features. Most-frequent or constant imputation can handle categories. The rule must be learned from training rows only.

### Business Example
An employee has a missing salary. The median is less sensitive to very high salaries than the mean. A missing department can be marked Unknown. These choices preserve the employee record.

### AI/ML Use
Many estimators cannot accept NaN values. Imputation makes the features usable. Different feature types should use different strategies. Avoid fitting an imputer on test data.

### Python Example

In [13]:
imp = SimpleImputer(strategy="median")
print(imp.fit_transform(df[["Salary"]]).ravel())

[25000. 52000. 52000. 75000. 40000. 62000.]


### Explanation
The median of observed salaries replaces the missing salary. The imputer remembers that median after fitting. Later datasets use the same stored median. This prevents inconsistent filling.

## 7. Scaling Pipeline

### Definition
Scaling changes the numeric scale of features. StandardScaler uses training mean and standard deviation. MinMaxScaler maps values to a chosen interval. Outliers can strongly affect both methods.

### Business Example
Salaries are measured in thousands while ages are smaller. A distance calculation may be dominated by salary. Scaling balances their numerical magnitudes. It does not make them equally important automatically.

### AI/ML Use
KNN, SVM, and gradient-based models often need scaling. Tree models are generally less sensitive. Scaling should happen after imputation. The scaler must fit only on training rows.

### Python Example

In [15]:
scaler = StandardScaler()
print(scaler.fit_transform(df[["Age"]].dropna()).round(2))

[[-1.28]
 [-0.33]
 [ 1.58]
 [-0.6 ]
 [ 0.63]]


### Explanation
The age values are centered and rescaled. The transformed values can be negative or positive. Negative does not mean an invalid age. It means the age is below the fitted mean.

## 8. Encoding Pipeline

### Definition
Encoding turns categorical labels into numeric representations. One-hot encoding avoids artificial ranking for nominal categories. Ordinal encoding is appropriate when categories have a real order. The chosen method affects model behavior.

### Business Example
An HR dataset includes IT, HR, and Sales departments. Assigning 1, 2, and 3 may imply a false order. One-hot encoding gives each department its own column. That is more appropriate for nominal labels.

### AI/ML Use
Most sklearn estimators require numeric inputs. One-hot encoding makes text categories usable. New labels must be handled deliberately. Extremely many categories can create many features.

### Python Example

In [17]:
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False) if "sparse_output" in OneHotEncoder().get_params() else OneHotEncoder(handle_unknown="ignore", sparse=False)
print(enc.fit_transform(df[["Department"]].fillna("Unknown")))

[[0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]]


### Explanation
The output contains zeros and ones. Each column represents a department category. No artificial numeric ranking is introduced. The encoder remembers the categories seen during fitting.

## 9. Combining Multiple Transformations

### Definition
A complete preprocessor can process many columns together. Numerical features follow imputation then scaling. Categorical features follow imputation then encoding. ColumnTransformer merges both outputs.

### Business Example
An employee record contains age, salary, city, and department. The first two need numerical processing. The others need categorical processing. A single object keeps these rules organized.

### AI/ML Use
A combined preprocessor creates model-ready features. It can be included inside a model Pipeline. It reduces training-serving mismatch. It is safer when fitted on training data only.

### Python Example

In [19]:
preprocessor = ColumnTransformer([("numeric", num_pipe, ["Age", "Salary"]), ("categorical", cat_pipe, ["City", "Department"])])
print(preprocessor.fit_transform(df).shape)

(6, 10)


### Explanation
The output has six rows and multiple encoded columns. Numerical and categorical pipelines run separately. Their results are combined in a single matrix. Column counts can increase after one-hot encoding.

## 10. fit()

### Definition
`fit()` learns parameters from the provided dataset. An imputer learns a median or most common value. A scaler learns mean and standard deviation. An encoder learns available categories.

### Business Example
A business trains a preprocessing system using historical employees. It learns typical salaries and departments. New employees should not change those stored training values. That keeps predictions consistent.

### AI/ML Use
Fit on the training set only. Do not fit on validation or test data. Otherwise information leaks from evaluation data. The fitted object can later transform unseen rows.

### Python Example

In [21]:
sample_imputer = SimpleImputer(strategy="median")
sample_imputer.fit(df[["Age"]])
print("Learned median:", sample_imputer.statistics_)

Learned median: [31.]


### Explanation
The imputer learns the median age. The result is stored in `statistics_`. No transformed array is returned by `fit()`. The same statistic is used during future transforms.

## 11. transform()

### Definition
`transform()` applies previously learned rules. It does not relearn the median or scaling parameters. It requires the transformer to be fitted first. It is used for validation, test, and production data.

### Business Example
A new employee has a missing age. The company applies the median learned from training records. It does not calculate a new median using that one employee. This ensures consistent processing.

### AI/ML Use
Test data must be transformed with the training-fitted object. This is essential for fair evaluation. The feature layout stays consistent. It also helps prevent leakage.

### Python Example

In [23]:
new_ages = pd.DataFrame({"Age": [22, np.nan, 48]})
print(sample_imputer.transform(new_ages))

[[22.]
 [31.]
 [48.]]


### Explanation
The missing age is replaced with the previously learned median. Existing ages stay unchanged. The imputer does not fit again. This is the correct approach for new records.

## 12. fit_transform()

### Definition
`fit_transform()` learns and applies transformations in one call. It is convenient when preparing training features. It is not appropriate for independently fitting test data. Pipelines also use this behavior internally.

### Business Example
A business prepares its historical training records. It learns the salary median and fills missing salaries immediately. New records later use only `transform()`. This avoids recalculating the rule.

### AI/ML Use
Use `fit_transform()` on training features. Use `transform()` on validation and test features. This distinction prevents evaluation leakage. It keeps model inputs consistent.

### Python Example

In [25]:
fresh_imputer = SimpleImputer(strategy="median")
print(fresh_imputer.fit_transform(df[["Age"]]))

[[24.]
 [31.]
 [31.]
 [45.]
 [29.]
 [38.]]


### Explanation
The call learns the median and fills missing ages. It combines two operations in one line. The fitted object can now transform new ages. Never independently fit a new imputer on test data.